# 🧠 Entrenamiento del Clasificador de Senas LSC

En este notebook entrenaremos la Red Neuronal Densa (MLP) para identificar las 20 señas del vocabulario LSC a partir de los 63 keypoints de MediaPipe.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import json
import os
import time

## 1. Carga de Datos

In [ ]:
X_train = np.load('../datasets/X_train.npy')
X_test = np.load('../datasets/X_test.npy')
y_train = np.load('../datasets/y_train.npy')
y_test = np.load('../datasets/y_test.npy')

with open('../datasets/label_encoder.json', 'r') as f:
    label_map = json.load(f)
    classes = list(label_map.keys())

print(f"X_train shape: {X_train.shape}")
print(f"Num clases: {len(classes)}")

## 2. Arquitectura de la Red Neuronal

Diseñamos una MLP con Dropout para evitar el sobreajuste (overfitting).

In [ ]:
model = models.Sequential([
    layers.Input(shape=(63,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation='softmax')
])

model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary()

## 3. Entrenamiento

In [ ]:
history = model.fit(
    X_train, y_train, 
    epochs=100, 
    batch_size=32, 
    validation_data=(X_test, y_test),
    callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

## 4. Matriz de Confusión

Permite ver qué señas confunde el modelo.

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Realidad')
plt.show()

## 5. Tiempo de Inferencia

El objetivo es que sea < 5ms por muestra.

In [ ]:
sample = X_test[0:1]
start_time = time.time()
for _ in range(100): 
    _ = model.predict(sample, verbose=0)
end_time = time.time()

avg_inference = (end_time - start_time) / 100 * 1000
print(f"Tiempo promedio de inferencia: {avg_inference:.2f} ms")